In [0]:
from pyspark.sql.functions import *

In [0]:
erp_bronze=spark.read.format("delta")\
    .load("s3://retail-lakehouse-ashu/bronze/erp/")

In [0]:
erp_bronze.groupBy(col("product_id")).count().filter(col("count") > 1).display()

In [0]:
erp_bronze.groupBy(col("category.category_id")).count().filter(col("count") > 1).display()

In [0]:
erp_bronze.groupBy(col("supplier.supplier_id")).count().filter(col("count") > 1).display()

In [0]:
erp_bronze.groupBy(col("category.category_id").alias("category_id")) \
    .agg(
        countDistinct("category.category_name").alias("name_count")
    ) \
    .filter(col("name_count") > 1) \
    .show()

In [0]:
erp_bronze.groupBy(
    col("supplier.supplier_id").alias("supplier_id")
).agg(
    countDistinct("supplier.supplier_name").alias("name_count"),
    countDistinct("supplier.supplier_city").alias("city_count"),
    countDistinct("supplier.supplier_rating").alias("rating_count")
).filter(
    (col("name_count") > 1) |
    (col("city_count") > 1) |
    (col("rating_count") > 1)
).show(truncate=False)

In [0]:
erp_bronze.printSchema()

In [0]:
dim_product_final=erp_bronze.select(col("product_id"),col("category.category_id").alias("category_id"),col("supplier.supplier_id").alias("supplier_id"),col("product_name"),col("status"),col("pricing.cost_price").alias("cost_price"),col("pricing.selling_price").alias("selling_price"),col("created_date").cast("date"),col("updated_date").cast("date"),col("ingestion_timestamp"),col("batch_id"))

In [0]:
dim_product_final.printSchema()

In [0]:
dim_category=erp_bronze.select(col("category.category_id").alias("category_id"),col("category.category_name").alias("category_name"),col("batch_id"))
dim_category_final=dim_category.dropDuplicates(["category_id"])

In [0]:
dim_supplier=erp_bronze.select(col("supplier.supplier_id").alias("supplier_id"),col("supplier.supplier_name").alias("supplier_name"),col("supplier.supplier_city").alias("supplier_city"),col("supplier.supplier_rating").alias("supplier_rating"),col("batch_id"))
dim_supplier_final=dim_supplier.dropDuplicates(["supplier_id"])

In [0]:
input_data=erp_bronze.count()
print("erp_bronze_input_data",input_data)
output_dim_product=dim_product_final.count()
print("output_dim_product",output_dim_product)
duplicate_removed_dim_product=input_data-output_dim_product
print("duplicate_removed_dim_product",duplicate_removed_dim_product)

In [0]:
input_category=dim_category.count()
print("input_category",input_category)
output_dim_category=dim_category_final.count()
print("output_dim_category",output_dim_category)
duplicate_removed_dim_category=input_category-output_dim_category
print("duplicate_removed_dim_category",duplicate_removed_dim_category)

In [0]:
input_supplier=dim_supplier.count()
print("input_supplier",input_supplier)
output_supplier=dim_supplier_final.count()
print("output_supplier",output_supplier)
duplicate_removed_supplier=input_supplier-output_supplier
print("duplicate_removed_supplier",duplicate_removed_supplier)

In [0]:
dim_product_final.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/erp/dim_product/")

In [0]:
dim_category_final.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/erp/dim_category/")

In [0]:
dim_supplier_final.write.mode("overwrite").format("delta").save("s3://retail-lakehouse-ashu/silver/erp/dim_supplier/")